In [1]:
# The goal is to demonstrate how services.xml variants work with single-node deployments with variations on which schemas are deployed.
# Start 3 Vespa dockers on separate ports
# Deploy to (1) with 'deploy:instance=local1', to another with 'deploy:instance=local2',
# List which indices were created by executing ls commands inside the docker containers
# Mention what happens if there is no `deploy:instance="default"` provided (the 3rd container)
# Shutdown all 3 docker images

In [2]:
%%bash
docker run \
  --detach \
  --rm \
  --name vespa-demo-1 \
  --publish 0.0.0.0:8080:8080 \
  --publish 0.0.0.0:19050:19050 \
  --publish 0.0.0.0:19071:19071 \
  vespaengine/vespa:8.682.72

4696c783cb5e7d6ac8744be999f869303de36d3e2c278a85f19adc34c0000199


In [3]:
%%bash
docker run \
  --detach \
  --rm \
  --name vespa-demo-2 \
  --publish 0.0.0.0:8081:8080 \
  --publish 0.0.0.0:19051:19050 \
  --publish 0.0.0.0:19072:19071 \
  vespaengine/vespa:8.682.72

d534f23b14e2a36fb8ccc763f0bad82e0e2e710a8502df67157617f2ff65cace


In [4]:
%%bash
docker run \
  --detach \
  --rm \
  --name vespa-demo-3 \
  --publish 0.0.0.0:8082:8080 \
  --publish 0.0.0.0:19052:19050 \
  --publish 0.0.0.0:19073:19071 \
  vespaengine/vespa:8.682.72

574a163b7a8c05ef1f1f6617be2ca0b5873edd88b80c6fc9668066df243d70e4


In [5]:
# Manually create an application package by zipping the files
!zip -r application.zip .

updating: services.xml (deflated 61%)
updating: schemas/ (stored 0%)
updating: schemas/doc2.sd (deflated 34%)
updating: schemas/doc1.sd (deflated 33%)
updating: README.md (deflated 78%)
updating: .vespaignore (deflated 23%)
updating: session.ipynb (deflated 77%)
  adding: session-vespa-cli.ipynb (deflated 77%)


In [6]:
%%bash
vespa deploy --application "default.default.local-1" -t "http://localhost:19071"

Success: Deployed '.' with session ID 2


In [7]:
%%bash
vespa deploy --application "default.default.local-2" -t "http://localhost:19072"

Success: Deployed '.' with session ID 2


In [8]:
%%bash
vespa deploy -t "http://localhost:19073"

Error: invalid application package (status 400)
Invalid application:
The <documents> element is mandatory in content cluster 'content'


CalledProcessError: Command 'b'vespa deploy -t "http://localhost:19073" \n'' returned non-zero exit status 1.

In [17]:
# ^ see that the application fails to deploy because no `<documents>` tag is provided.
# This is because an implicit `instance=default` is used by the configserver if there a siblings that has more specific variants.

In [9]:
# Check which schemas are deployed to the instance=local-1
!docker exec vespa-demo-1 sh -c "ls /opt/vespa/var/db/vespa/search/cluster.content/n0/documents"

doc1


In [11]:
# As expected, only the `doc1`

In [10]:
# Check which schemas are deployed to the instance=local-2
!docker exec vespa-demo-2 sh -c "ls /opt/vespa/var/db/vespa/search/cluster.content/n0/documents"

doc1
doc2


In [15]:
# We have `doc1` and the `doc2` schemas deployed. Nice.

In [11]:
!docker kill vespa-demo-3
!docker kill vespa-demo-2
!docker kill vespa-demo-1

vespa-demo-3
vespa-demo-2
vespa-demo-1
